# Critical energy infrastructure in Denmark under attacks


Modern power systems are large-scale, highly interconnected networks where failures can propagate rapidly across regions. Identifying critical components—nodes or transmission elements whose failure leads to severe system imbalance—is essential for ensuring grid reliability and resilience.

This project performs a systematic vulnerability analysis of the Danish transmission grid by combining:

 - Network modeling
 - Monte Carlo scenario generation 
 - Optimization-based power flow simulation
 - Graph Neural Networks (GNNs) for prediction

The objective is to quantify the impact of component failures and identify the most critical nodes and edges under varying system conditions.


<div style="display: flex; gap: 20px;">
    <div style="text-align: center;">
        <img src="fullDanishGrid1.png" width="350">
        <p><b>(a)</b> Transmission network</p>
    </div>
    <div style="text-align: center;">
        <img src="fullDanishGrid2.png" width="350">
        <p><b>(b)</b> Transmission network without powerplants and substations </p>
    </div>
</div>


**Figure a and b:** Base representation of the Danish transmission network. Nodes represent substations and are color-coded by voltage level, while edges represent transmission lines and cables. The network illustrates the hierarchical structure of the grid, including high-voltage backbone connections, regional substations, and cross-border HVDC links. This serves as the baseline for the subsequent criticality analysis.

Node and edge colors indicate voltage levels and infrastructure types within the transmission network. 

Red represents high-voltage (400 kV) substations and lines, forming the backbone of the Danish transmission system. Green denotes medium-voltage (220 kV) infrastructure, while black corresponds to lower transmission levels (150/132 kV). 

Special components are highlighted separately: HVDC connections and converter stations are shown in blue, offshore wind connections in yellow, and conventional power plants are marked with lightning symbols.

Edges follow the same color convention as nodes to maintain consistency across voltage levels. Additionally, dashed lines represent underground or submarine cables, while solid lines indicate overhead transmission lines.




## Results
**Figure a and b below:** Visualization of the most critical components in the Danish transmission network, based on the ten highest-impact failure scenarios. Figure a shows the most critical point after one edge/node removal. Figure b shows the unmet demad for both node and edge removals simultaneously.





<div style="display: flex; gap: 20px;">
    <div style="text-align: center;">
        <img src="1nodeedgeremoval.png" width="350">
        <p><b>(a)</b> One edge/node removal</p>
    </div>
    <div style="text-align: center;">
        <img src="gnn_Results.png" width="450">
        <p><b>(b)</b> GNN node/edge removals </p>
    </div>
</div>



<div style="display: flex; gap: 20px;">
    <div style="text-align: center;">
        <img src="barChart1.png" width="650">
        <p><b>(a)</b> One edge/node removal</p>
    </div>
    <div style="text-align: center;">
        <img src="barChartgnn.png" width="300">
        <p><b>(b)</b> GNN node/edge removals </p>
    </div>
</div>

Libaries:

In [ ]:
import ast
import numpy as np
import torch
import pandas as pd
import cleaning
import Evaluation
from GNN import (
    train_gnn, _build_sample_dataset, evaluate_test,
    predict_criticality, N_NUMERIC, N_SOURCES,
)
from scenarios import get_all_mean_scenarios, SCENARIOS, apply_scenario_mean

## Dataset
The dataset is provided by Energinet and contains detailed structural and operational information about the Danish transmission system, including:

 - Substations (Bus sheet)
 - Transmission lines (Line sheet)
 - Generators (Generator sheet)
 - Loads (Load sheet)
 - HVDC interconnectors (International connections) (HVDC sheet)
 - Transformers (Transformer2 and Transformer3 sheets)

## Data cleaning
The system is modeled as a graph:

Nodes (buses):
- Represent substations
- Attributes include:
    - supply (MW)
    - demand (MW)
    - generation limits (p_min, p_max)
    - energy source (e.g., wind, gas, hydro)
- Cordinates (added manually)

Edges:
- Represent transmission lines and transformers
- Include capacity constraints derived from:
$$
\text { capacity } \approx \sqrt{3} \cdot V \cdot I
$$
- Transformer connections are explicitly included to ensure topological completeness
- Some lines where added manually. (Storebælt and Cobra international line)

### Disaggregated vs Aggregated Graphs

The model distinguishes between two graph representations:

#### Disaggregated graph
Multiple nodes per physical bus or node (one per generation source)
Internal edges connect the subcomponents
Used to capture generation flexibility on source level (Wind, Solar, Gas, etc.)
#### Aggregated graph
One node per bus
Aggregates:
- total supply
- total demand
flexibility (p_addable, p_removable)
Used for:
- optimization model
- attack simulations

*Please see all the detail of the cleaning functions in the cleaning.py file

In [ ]:
file_path = "publicdataexportv131450706334_with_lon_lat.xlsx"

# Loading sheets
bus = pd.read_excel(file_path, sheet_name="Bus", header=3)
line = pd.read_excel(file_path, sheet_name="Line", header=3)
gen = pd.read_excel(file_path, sheet_name="Generator", header=3)
load = pd.read_excel(file_path, sheet_name="Load", header=3)
hvdc = pd.read_excel(file_path, sheet_name="HVDC", header=3)
transformer2 = pd.read_excel(file_path, sheet_name="Transformer2", header=3)
transformer3 = pd.read_excel(file_path, sheet_name="Transformer3", header=3)

for df in [bus, line, gen, load, hvdc, transformer2, transformer3]:
    df.columns = [str(c).strip() for c in df.columns]


# This is our main cleaning function that turns our raw data into an unaggregated graph, that is used for the scenario generation
g_unaggregated = cleaning.main_clean(bus, line, gen, load, hvdc, transformer2, transformer3)
edges = pd.read_excel('danish_grid_graph_ready.xlsx', sheet_name='edges')
nodes = pd.read_excel('danish_grid_graph_ready.xlsx', sheet_name='nodes')

### Graph visualization
Below we visualize the graph using the built-in function of our graph tool networkx

In [ ]:
cleaning.print_graph(cleaning.aggregate_graph(g_unaggregated, edges))

## Optimization Model

Each failure scenario is evaluated using a linear optimization model based on power flow and load shedding


Decision Variables
- $t_{u v}$ : power flow on each line
- $s_{\text {pos }}$ : oversupply (curtailment)
- $s_{n e g}$ : unmet demand (load shedding)
- $\delta_{\text {add}}, \delta_{\text {rem }}$ : generation adjustments, these are calculated during the graph aggregation. We assume gas is dynamic, hydro can be increased/decreased by 20%, and wind can be turned off 

Objective Function
Minimize total system imbalance:

$$
\min \sum_n\left(s_{p o s, n}+s_{n e g, n}\right)
$$

Constraints
1. Flow balance at each node:

$$
\text { supply }+\delta_{a d d}-\delta_{\text {rem}}-\text { demand }+ \text { inflow }- \text { outflow }-s_{\text {pos }}+s_{n e g}=0
$$

2. Transmission capacity:

$$
-C_{u v} \leq t_{u v} \leq C_{u v}
$$

3. Generation flexibility limits
$$
\delta_{\text {add }} \leq C_{add} \\
\delta_{\text {rem }} \leq C_{rem}
$$

*This model is used for the attack single-removal simulation, greedy attack simulation and data generation for the GNN, see below

## Scenario-conditioned one-removal attack simulations

Only balanced scenarios (supply ≈ demand after rebalancing) are valid baselines for attack evaluation.
If the grid already has unmet demand before an attack, the simulation results cannot isolate the attack's impact.

Only balanced scenarios are considered valid:

$$
\text { gap }=\mid \text { supply }- \text { demand } \mid
$$


A scenario is accepted if:

$$
\frac{\text { gap }}{\text { demand }} \leq 1 \%
$$


This ensures that any imbalance observed during simulations is caused only by the simulated failure, not by initial conditions.

Also for the one-removal attack simulation we only use the mean scenarios, as we have limited cpmputational resource, as each scenario evaluation takes about 1-3 min depending on laptop power.

In [ ]:
mean_scenarios = get_all_mean_scenarios(base_scenario=g_unaggregated)

### All single removals simulation
Getting all single removals of nodes and edges for each mean scenario.

In total we have 9 balanced mean scenarios

For each valid scenario, we simulate single-component failures:
- Node removals (substation failures)
- Edge removals (line or transformer outages)

For each scenario, we remove each node and edge once and run the optimization model to get the grid imbalance. We then will know for each scenario, aht the best aingle attack is for the grid


### Performance Metrics

Each attack is evaluated using:

- Total unmet demand (MW) → primary indicator of failure severity
- Total oversupply (MW) → stranded generation
- Number of connected components → grid fragmentation

### Assessment

Each node/edge is assigned a criticality score based on:

system imbalance after removal
structural impact on connectivity

Initially, results are ranked directly by unmet demand.

Later, a normalized weighted score is used:
$$
\text { score }=0.7 \cdot \text { unmet }_{\text {norm }}+0.2 \cdot \text { oversupply }_{\text {norm }}+0.1 \cdot \text { groups }_{\text {norm }}
$$
This ensures comparability across metrics.


In [ ]:
scenario_results_list = []

for G in mean_scenarios:
    print("-> ", G.name, " running...")
    df = Evaluation.simulation_all_single_removals(cleaning.aggregate_graph(G, edges))
    scenario_results_list.append(df)

scenario_results = pd.concat(scenario_results_list, ignore_index=True)
scenario_results.to_excel("scenario_results.xlsx", index=False)
print(f"\nSaved {len(scenario_results)} rows to scenario_results.xlsx")

In [ ]:
scenario_results.sort_values('total_unmet_demand_MW', ascending=False).head(20)

We can see xxx

## Greedy attack heuristic
It is interesting to see the worst performing nodes and edges, but in reality we can expect an attack to involve multiple nodes and edges that get deactivated at once or in a short sequence. 

However running a similar optimization model as before with multiple k removals would mean anexploding solution space.

Finding the optimla attack sequence for k number of attacks would mean around $(n_{nodes} + n_{edges})^k$ mathematical model calls.
For k=2 or k=3 this might still be viable using high performance computing and parallelization, but with the exponential nature of this at k=4 there will already be 62.5 billion calls necessary.

Therefore we decided to implement a simple greedy heuristic to see whats possible.
It takes a graph and for each step will pick the best candiate for removal.

In [ ]:
for G in mean_scenarios:
    res = Evaluation.greedy_sequential_attack(G)
    # Shouldn't it be like this instead? Like above. Or does greedy take unaggregated graph as input?
    #res = Evaluation.greedy_sequential_attack(cleaning.aggregate_graph(G, edges))
    # print() TODO


## Attack Evaluation using a GNN
From our previous work we can see that the evaluation of an attack and how severe it is, is possible only for smaller attacks. We can fnd out a good sequential attack sequence using a greedy heuristic, but it only encompasses a mini-subset of attacksans doesn't tell us anything about the severity of other larger attacks.

This is were our GNN model comes into play. We use our mathematical model combined with the Monte-Carlo scenario generation to generate a data base for the training of the GNN. 
- First we generate scenarios from our Monte Carlo scenario generation
- Then we use this to simulate attacks with different number of attacks (k=1-6) and use a closeness factor to generate more scenarios that are close together, so that the GNN learns better, what it means if two close nodes or edges are cut.
- Lastly we run the mathematical model on these generated graphs with attacks and calulate the unmet demand, which is our predictor variable

We used a multi-threaded script to generate around 190k unique rows of data

The goal of the GNN is to give it a graph that already is missing some nodes or edges and it predicts the unmet demand of that graph

### Prepare the data
First we prepare the data for the GNN. We use a 70/20/10 split for train/val/test.

In [ ]:
### Reuse cleaned data from earlier in the notebook
g_disagg = g_unaggregated
edges_df = edges

### Simulation results from the math mod results (no header)
CSV_PATH  = "results/simulation_20260502_000845.csv"
COL_NAMES = ["scenario_id", "k", "removed_nodes", "removed_edges",
                "total_unmet_demand_MW", "total_oversupply_MW", "num_groups"]
results_df = pd.read_csv(CSV_PATH, header=None, names=COL_NAMES)
print(f"Loaded {len(results_df):,} rows covering "
        f"{results_df['scenario_id'].nunique()} scenarios.")

### Canonical deduplication (must happen BEFORE splitting)
# Same removal set in different list orders (e.g. [102,103] vs [103,102])
# builds the same perturbed graph → keep only first occurrence.
# Deduping before split prevents the same removal set appearing in both
# train and test, which would be data leakage.
def _canon_key(rn, re):
    nodes = sorted(ast.literal_eval(str(rn)))
    edges = sorted(tuple(sorted(e)) for e in ast.literal_eval(str(re)))
    return repr(nodes) + "|" + repr(edges)

results_df["_key"] = (results_df["scenario_id"] + "||" + results_df.apply(lambda r: _canon_key(r["removed_nodes"], r["removed_edges"]), axis=1))
results_df = (results_df.drop_duplicates(subset="_key").drop(columns="_key").reset_index(drop=True))
print(f"After dedup: {len(results_df):,} unique samples.")

### Build aggregated scenario graphs
available_ids   = set(results_df["scenario_id"].unique())
scenario_graphs = {}
for sid, scenario in SCENARIOS.items():
    if sid not in available_ids:
        continue
    G_sc = apply_scenario_mean(g_disagg, scenario)
    scenario_graphs[sid] = cleaning.aggregate_graph(G_sc, edges_df)

### Stratified 70/20/10 row split (train/val/test)
# Stratify by scenario so every scenario is proportionally represented in all three splits.
# Seed=1 since we are group 1 :)
rng = np.random.default_rng(1)
train_rows, val_rows, test_rows = [], [], []
for _, grp in results_df.groupby("scenario_id"):
    idx     = rng.permutation(len(grp))
    n_train = int(len(grp) * 0.70)
    n_val   = int(len(grp) * 0.20)
    train_rows.append(grp.iloc[idx[:n_train]])
    val_rows.append(grp.iloc[idx[n_train:n_train + n_val]])
    test_rows.append(grp.iloc[idx[n_train + n_val:]])

train_df = pd.concat(train_rows).reset_index(drop=True)
val_df   = pd.concat(val_rows).reset_index(drop=True)
test_df  = pd.concat(test_rows).reset_index(drop=True)
print(f"\nSplit (stratified by scenario, seed=1):")
print(f"  train: {len(train_df):,}  val: {len(val_df):,}  test: {len(test_df):,}")

### Train the GNN
We train the GNN on the train data. We use the validation data for choosing the best model (lowest error on validation data) and for implementing early stopping.

In [ ]:
model, t_mean, t_std = train_gnn(
    scenario_graphs, train_df, val_df=val_df,
    epochs=300, lr=1e-3, hidden=64,
    eval_every=1, patience=10,
    batch_size=32, pos_weight=3.0,
)

### Save model
torch.save({"model_state": model.state_dict(), "in_feats": N_NUMERIC + N_SOURCES, "t_mean": t_mean, "t_std": t_std}, "gnn_model.pt")
print("Model saved → gnn_model.pt")

### Test
We test the GNN on the test data

In [ ]:
test_samples = _build_sample_dataset(test_df, scenario_graphs)
test_results = evaluate_test(model, test_samples, t_mean, t_std)
test_results.to_csv("gnn_test_results.csv", index=False)
print("Test predictions saved → gnn_test_results.csv")


### k=1 criticality ranking on first available scenario
first_sid = sorted(scenario_graphs.keys())[0]
preds = predict_criticality(model, scenario_graphs[first_sid], t_mean, t_std)
print(f"\nTop 15 critical elements (k=1) in {first_sid}:")
print(preds.head(15).to_string(index=False))
preds.to_excel("gnn_predictions.xlsx", index=False)
print("k=1 predictions saved → gnn_predictions.xlsx")

In [ ]:
results = pd.read_csv('gnn_test_results.csv')

In [14]:
mae  = np.mean(np.abs(results["predicted_unmet_MW"] - results["true_unmet_MW"]))
rmse = np.sqrt(np.mean((results["predicted_unmet_MW"] - results["true_unmet_MW"]) ** 2))
avg  = results["true_unmet_MW"].mean()
print(f"Overall (n={len(results):,}, avg true={avg:.1f} MW):")
print(f"  MAE:  {mae:.1f} MW")
print(f"  RMSE: {rmse:.1f} MW")

nz = results[results["true_unmet_MW"] > 0]
mae_nz  = np.mean(np.abs(nz["predicted_unmet_MW"] - nz["true_unmet_MW"]))
rmse_nz = np.sqrt(np.mean((nz["predicted_unmet_MW"] - nz["true_unmet_MW"]) ** 2))
avg_nz  = nz["true_unmet_MW"].mean()
print(f"\nNon-zero only (n={len(nz):,}, avg true={avg_nz:.1f} MW):")
print(f"  MAE:  {mae_nz:.1f} MW")
print(f"  RMSE: {rmse_nz:.1f} MW")

Overall (n=19,358, avg true=86.9 MW):
  MAE:  8.2 MW
  RMSE: 23.5 MW

Non-zero only (n=5,489, avg true=306.5 MW):
  MAE:  14.4 MW
  RMSE: 39.1 MW


We see that our GNN is able to approximate the simulateed data well and is able to predict the expected unmet demand to a good extent.

We now would like to investigate how good it is at actually detecting a big attack and how many times it predicts bad attack. 

In [20]:
THRESHOLD = 600  # MW

# Accuracy on genuinely severe attacks
severe = results[results["true_unmet_MW"] > THRESHOLD]
mae_s  = np.mean(np.abs(severe["predicted_unmet_MW"] - severe["true_unmet_MW"]))
rmse_s = np.sqrt(np.mean((severe["predicted_unmet_MW"] - severe["true_unmet_MW"]) ** 2))
print(f"Severe attacks (true > {THRESHOLD} MW, n={len(severe):,}, avg true={severe['true_unmet_MW'].mean():.1f} MW):")
print(f"  MAE:  {mae_s:.1f} MW")
print(f"  RMSE: {rmse_s:.1f} MW")

# False alarms: predicted severe but true is low
false_alarms = results[(results["predicted_unmet_MW"] > THRESHOLD) & (results["true_unmet_MW"] <= THRESHOLD)]
print(f"\nFalse alarms — predicted > {THRESHOLD} MW but true <= {THRESHOLD} MW (n={len(false_alarms):,}, {len(false_alarms)/len(results)*100:.1f}% of all predictions):")
print(f"  Avg predicted: {false_alarms['predicted_unmet_MW'].mean():.1f} MW  |  Avg true: {false_alarms['true_unmet_MW'].mean():.1f} MW")

# Missed attacks: true is severe but predicted low
missed = results[(results["predicted_unmet_MW"] <= THRESHOLD) & (results["true_unmet_MW"] > THRESHOLD)]
print(f"\nMissed attacks — true > {THRESHOLD} MW but predicted <= {THRESHOLD} MW (n={len(missed):,}, {len(missed)/len(severe)*100:.1f}% of severe attacks):")
print(f"  Avg predicted: {missed['predicted_unmet_MW'].mean():.1f} MW  |  Avg true: {missed['true_unmet_MW'].mean():.1f} MW")

Severe attacks (true > 600 MW, n=271, avg true=833.2 MW):
  MAE:  34.7 MW
  RMSE: 85.7 MW

False alarms — predicted > 600 MW but true <= 600 MW (n=23, 0.1% of all predictions):
  Avg predicted: 613.0 MW  |  Avg true: 557.7 MW

Missed attacks — true > 600 MW but predicted <= 600 MW (n=19, 7.0% of severe attacks):
  Avg predicted: 487.8 MW  |  Avg true: 714.1 MW


Explanation....
We now want to also show what edges and nodes are among the most vulnerable.

Am thinking a graph similar to the top

## Conclusion
### Highlights
  - **Optimization-based attack evaluation.** Each removal is scored with a linear program that minimizes total system imbalance (unmet demand + excess generation) under flow balance, capacity, and flexibility constraints, a physically grounded severity metric rather than a structural proxy.
 
  - **GNN with dual node and edge criticality heads.** A two-layer GCN predicts unmet demand for both node and edge removals simultaneously, learning from grid topology and operational attributes. It generalizes across scenarios and scores any component in near real-time.
**updated after GNN is finished**
  
 
  - **12 calibrated stress scenarios.** The set spans the realistic operating envelope: high-wind surplus, low-wind import dependency, winter peak, storm with derated lines, Storeb{\ae}lt outage, targeted HVDC cut. 

  - **Correlated Monte Carlo sampling.** Scenario uncertainty is propagated through 500 draws using a Gaussian copula: wind offshore/onshore, demand and thermal backup, and HVDC availability are sampled jointly with calibrated correlations, producing more realistic joint distributions than independent sampling.
 
  - **Full single-removal attack surface.** Every node and edge was removed under every balanced scenario, producing thousands of (scenario, removal) pairs -- both the training signal for the GNN and a direct, exhaustive criticality ranking.


### Limitations
- **Static 2020 snapshot.** The grid data reflects a single
point in time; infrastructure changes since 2020 are not
captured.
- **Supply and demand are not dynamic.** Generation and
load are fixed values – they do not vary throughout the
day, across seasons, or in response to market signals.
- **No AC/DC power-flow physics.** The model uses graph
topology and capacity accounting, not load-flow equa-
tions. Frequency stability, voltage constraints, and relay
cascades are outside scope.
- **Generator ramp constraints ignored.** Thermal and
gas plants are treated as instantly dispatchable, likely
underestimating short-term unmet demand after a sudden
generation loss.


## Bibliography

[1] Data.gov, *MTA Dataset Hourly Ridership 2017-2019*, [https://catalog.data.gov/dataset/mta-subway-hourly-ridership-2017-2019](https://catalog.data.gov/dataset/mta-subway-hourly-ridership-2017-2019)

[2] Visual Crossing, *Weather Data Services*, [https://www.visualcrossing.com/](https://www.visualcrossing.com/)

[3] IEA, *Energy Mix of Denmark*, [https://www.iea.org/countries/denmark/energy-mix](https://www.iea.org/countries/denmark/energy-mix)